# Inference for APEBench GSDR variants
General pipeline: Load testset -> Inference
Since I am testing singular datasets, I'll just use PBDLDataModule instead of MixedDataModule. It is also possible to use MixedDataModule instead for a singular dataset.

Pretrained datasets used in this notebook:

gs_alpha/gs_beta/gs_gamma/gs_epsilon

## Note
I would recommend using PDEtransformer's script to generate APEBench data, specifically the low_res script.

Plotting script is custom.


# Loading of gs_alpha

In [ ]:
from pdetransformer.core.separate_channels import PDETransformer
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = PDETransformer.from_pretrained('thuerey-group/pde-transformer', subfolder='sc-s').to(device)

print(model)

In [ ]:
from pdetransformer.data.pbdl_dataloader.dataset import Dataset as PBDLDataset
import os
import certifi

raw_dir = '/Volumes/T7/APEBench/data'

params_test = {
    'dset_name': 'gs_theta',
    "local_datasets_dir": raw_dir,
    "global_dataset_dir": raw_dir,
    "dataset_ext": ".hdf5"
}

test_set = PBDLDataset(**params_test)

# Note
By default, normalize_const = 'None' for all gs variants. This is hardcoded. So the cell below putting normalize_const = 'mean-std' would not affect anything.

In [ ]:
from pdetransformer.data.pbdl_module import PBDLDataModule

raw_dir = '/Volumes/T7/APEBench/gs_alpha_all'

path_config = {'2D_APE_xxl': raw_dir}

dataset = PBDLDataModule(
    path_index=path_config,
    dataset_name='gs_alpha',
    dataset_type='2D_APE_xxl',
    unrolling_steps=1,
    test_unrolling_steps=29,
    batch_size=1,
    num_workers=0,
    cache_strategy='none',
    normalize_data= 'mean-std',
    normalize_const= 'mean-std'
)


dataset.setup(stage='test')
test_loader = dataset.test_dataloader()




# Prediction for gs_alpha

An error is thrown if I don't change the dtypes of the corresponding data below.

In [ ]:
from pdetransformer.core.separate_channels import Supervised
import torch
strategy = Supervised(model, timesteps=2)
data = next(iter(test_loader))
# print(data)
# The PDE index must be an integer
data['physical_metadata']['PDE'] = data['physical_metadata']['PDE'].long()

# The Field indices (channels) must be integers
data['physical_metadata']['Fields'] = data['physical_metadata']['Fields'].long()

data['physical_metadata']['Constants'] = data['physical_metadata']['Constants'].long()

prediction, reference = strategy.predict(data, device=device, num_frames=29)

# Plotting of pred and reference for gs_alpha

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

field = 'Concentration A'
dataset_name = 'gs_alpha'
time_steps = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

prediction_steps = prediction[0, :, :, :]
reference_steps = reference[0, :, :, :]

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)

fig, axes = plt.subplots(2, len(time_steps), figsize=(16.65,3.2), dpi=200)

vmin = reference_steps.min()
vmax = reference_steps.max()

for j, t in enumerate(time_steps):

    axes[0, j].imshow(prediction_steps[t][0],  cmap=cmap, vmin=vmin, vmax=vmax)

    axes[0, j].set_xticks([])
    axes[0, j].set_yticks([])

    axes[1, j].imshow(reference_steps[t][0], cmap=cmap,
                      vmin=vmin, vmax=vmax)

    axes[1, j].set_xticks([])
    axes[1, j].set_yticks([])

axes[0, 0].set_ylabel('Prediction', fontsize=12)
axes[1, 0].set_ylabel('Reference', fontsize=12)

for j, t in enumerate(time_steps):
    axes[1, j].set_xlabel(f't={t}', fontsize=12)

fig.text(0.8712, 0.85, f'{dataset_name}',
         fontsize=12, ha='right', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))

plt.subplots_adjust(wspace=0.03, hspace=0.03)

cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_label(field, fontsize=12)

plt.show()

# Loading of gs_beta

In [ ]:
from pdetransformer.data.pbdl_module import PBDLDataModule

raw_dir = '/Volumes/T7/APEBench/gs_beta_all'

path_config = {'2D_APE_xxl': raw_dir}

dataset = PBDLDataModule(
    path_index=path_config,
    dataset_name='gs_beta',
    dataset_type='2D_APE_xxl',
    unrolling_steps=1,
    test_unrolling_steps=29,
    batch_size=1,
    num_workers=0,
    cache_strategy='none',
    normalize_data= 'mean-std',
    normalize_const= 'mean-std'
)


dataset.setup(stage='test')
test_loader = dataset.test_dataloader()




# Prediction for gs_beta

In [ ]:
from pdetransformer.core.separate_channels import Supervised
import torch
strategy = Supervised(model, timesteps=2)
data = next(iter(test_loader))
print(data)

data['physical_metadata']['PDE'] = data['physical_metadata']['PDE'].long()
data['physical_metadata']['Fields'] = data['physical_metadata']['Fields'].long()
data['physical_metadata']['Constants'] = data['physical_metadata']['Constants'].long()

print(data)
prediction, reference = strategy.predict(data, device=device, num_frames=29)

# Plotting pred and reference for gs_beta

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

field = 'Concentration A'
dataset_name = 'gs_beta'
time_steps = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

prediction_steps = prediction[0, :, :, :]
reference_steps = reference[0, :, :, :]

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)

fig, axes = plt.subplots(2, len(time_steps), figsize=(16.65,3.2), dpi=200)

vmin = reference_steps.min()
vmax = reference_steps.max()

for j, t in enumerate(time_steps):

    axes[0, j].imshow(prediction_steps[t][0],  cmap=cmap, vmin=vmin, vmax=vmax)

    axes[0, j].set_xticks([])
    axes[0, j].set_yticks([])

    axes[1, j].imshow(reference_steps[t][0], cmap=cmap,
                      vmin=vmin, vmax=vmax)

    axes[1, j].set_xticks([])
    axes[1, j].set_yticks([])

axes[0, 0].set_ylabel('Prediction', fontsize=12)
axes[1, 0].set_ylabel('Reference', fontsize=12)

for j, t in enumerate(time_steps):
    axes[1, j].set_xlabel(f't={t}', fontsize=12)

fig.text(0.8712, 0.85, f'{dataset_name}',
         fontsize=12, ha='right', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))

plt.subplots_adjust(wspace=0.03, hspace=0.03)

cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_label(field, fontsize=12)

plt.show()

# Loading of gs_gamma

In [ ]:
from pdetransformer.data.pbdl_module import PBDLDataModule

raw_dir = '/Volumes/T7/APEBench/gs_gamma_all'

path_config = {'2D_APE_xxl': raw_dir}

dataset = PBDLDataModule(
    path_index=path_config,
    dataset_name='gs_gamma',
    dataset_type='2D_APE_xxl',
    unrolling_steps=1,
    test_unrolling_steps=29,
    batch_size=1,
    num_workers=0,
    cache_strategy='none',
    normalize_data= 'mean-std',
    normalize_const= 'mean-std'
)


dataset.setup(stage='test')
test_loader = dataset.test_dataloader()




# Prediction for gs_gamma

In [ ]:
from pdetransformer.core.separate_channels import Supervised
import torch
strategy = Supervised(model, timesteps=2)
data = next(iter(test_loader))
print(data)

data['physical_metadata']['PDE'] = data['physical_metadata']['PDE'].long()
data['physical_metadata']['Fields'] = data['physical_metadata']['Fields'].long()
data['physical_metadata']['Constants'] = data['physical_metadata']['Constants'].long()

print(data)
prediction, reference = strategy.predict(data, device=device, num_frames=29)

# Plotting pred and reference for gs_gamma

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

field = 'Concentration A'
dataset_name = 'gs_gamma'
time_steps = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

prediction_steps = prediction[0, :, :, :]
reference_steps = reference[0, :, :, :]

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)

fig, axes = plt.subplots(2, len(time_steps), figsize=(16.65,3.2), dpi=200)

vmin = reference_steps.min()
vmax = reference_steps.max()

for j, t in enumerate(time_steps):

    axes[0, j].imshow(prediction_steps[t][0],  cmap=cmap, vmin=vmin, vmax=vmax)

    axes[0, j].set_xticks([])
    axes[0, j].set_yticks([])

    axes[1, j].imshow(reference_steps[t][0], cmap=cmap,
                      vmin=vmin, vmax=vmax)

    axes[1, j].set_xticks([])
    axes[1, j].set_yticks([])

axes[0, 0].set_ylabel('Prediction', fontsize=12)
axes[1, 0].set_ylabel('Reference', fontsize=12)

for j, t in enumerate(time_steps):
    axes[1, j].set_xlabel(f't={t}', fontsize=12)

fig.text(0.8712, 0.85, f'{dataset_name}',
         fontsize=12, ha='right', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))

plt.subplots_adjust(wspace=0.03, hspace=0.03)

cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_label(field, fontsize=12)

plt.show()

# Loading of gs_epsilon

In [ ]:
from pdetransformer.data.pbdl_module import PBDLDataModule

raw_dir = '/Volumes/T7/APEBench/gs_epsilon_all'

path_config = {'2D_APE_xxl': raw_dir}

dataset = PBDLDataModule(
    path_index=path_config,
    dataset_name='gs_epsilon',
    dataset_type='2D_APE_xxl',
    unrolling_steps=1,
    test_unrolling_steps=29,
    batch_size=1,
    num_workers=0,
    cache_strategy='none',
    normalize_data= 'mean-std',
    normalize_const= 'mean-std'
)


dataset.setup(stage='test')
test_loader = dataset.test_dataloader()

# Prediction for gs_epsilon

In [ ]:
from pdetransformer.core.separate_channels import Supervised
import torch
strategy = Supervised(model, timesteps=2)
data = next(iter(test_loader))
print(data)

data['physical_metadata']['PDE'] = data['physical_metadata']['PDE'].long()
data['physical_metadata']['Fields'] = data['physical_metadata']['Fields'].long()
data['physical_metadata']['Constants'] = data['physical_metadata']['Constants'].long()

print(data)
prediction, reference = strategy.predict(data, device=device, num_frames=29)

# Plotting pred and reference for gs_epsilon

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

field = 'Concentration A'
dataset_name = 'gs_epsilon'
time_steps = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

prediction_steps = prediction[0, :, :, :]
reference_steps = reference[0, :, :, :]

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)

fig, axes = plt.subplots(2, len(time_steps), figsize=(16.65,3.2), dpi=200)

vmin = reference_steps.min()
vmax = reference_steps.max()

for j, t in enumerate(time_steps):

    axes[0, j].imshow(prediction_steps[t][0],  cmap=cmap, vmin=vmin, vmax=vmax)

    axes[0, j].set_xticks([])
    axes[0, j].set_yticks([])

    axes[1, j].imshow(reference_steps[t][0], cmap=cmap,
                      vmin=vmin, vmax=vmax)

    axes[1, j].set_xticks([])
    axes[1, j].set_yticks([])

axes[0, 0].set_ylabel('Prediction', fontsize=12)
axes[1, 0].set_ylabel('Reference', fontsize=12)

for j, t in enumerate(time_steps):
    axes[1, j].set_xlabel(f't={t}', fontsize=12)

fig.text(0.8712, 0.85, f'{dataset_name}',
         fontsize=12, ha='right', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))

plt.subplots_adjust(wspace=0.03, hspace=0.03)

cbar = fig.colorbar(axes[0, 0].images[0], ax=axes, orientation='vertical', fraction=0.02, pad=0.01)
cbar.set_label(field, fontsize=12)

plt.show()